In [1]:
import pandas as pd
from scipy.spatial.distance import pdist, cdist
import numpy as np
import math
import datetime as dt
from datetime import timedelta
import itertools
from collections import defaultdict
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "../tests")))

In [75]:
# sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
import filters
import daphmeIO as loader
import constants as constants
import stop_detection_modified as SD
import stop_detection as baseSD

In [76]:
path = '../data/sample4/'

traj_cols = {'user_id':'user_id',
             'latitude':'dev_lat',
             'longitude':'dev_lon',
             'datetime':'local_datetime'}

df = loader.from_file(path, traj_cols=traj_cols, format='csv')

In [77]:
df = filters.to_projection(df, latitude='dev_lat', longitude='dev_lon')

In [66]:
df['timestamp'] = df['local_datetime'].astype('int64') // 10**9

In [67]:
df

,user_id,dev_lat,dev_lon,local_datetime,x,y,timestamp
0,wizardly_joliot,38.321711,-36.667334,2024-01-01 14:29:00,-4.081789e+06,4.624973e+06,1704119340
1,wizardly_joliot,38.321676,-36.667365,2024-01-01 14:35:00,-4.081792e+06,4.624968e+06,1704119700
2,wonderful_swirles,38.321017,-36.667869,2024-01-01 15:06:00,-4.081849e+06,4.624874e+06,1704121560
3,youthful_galileo,38.321625,-36.666612,2024-01-01 08:47:00,-4.081709e+06,4.624961e+06,1704098820
4,youthful_galileo,38.321681,-36.666841,2024-01-01 09:59:00,-4.081734e+06,4.624969e+06,1704103140
...,...,...,...,...,...,...,...
25830,angry_spence,38.320399,-36.667438,2024-01-15 07:23:00,-4.081801e+06,4.624787e+06,1705303380
25831,angry_spence,38.320413,-36.667469,2024-01-15 07:29:00,-4.081804e+06,4.624789e+06,1705303740
25832,angry_spence,38.320384,-36.667455,2024-01-15 07:33:00,-4.081802e+06,4.624785e+06,1705303980
25833,angry_spence,38.320349,-36.667473,2024-01-15 07:39:00,-4.081804e+06,4.624780e+06,1705304340


In [68]:
sample_user_df = df[df['user_id'] == "angry_spence"]

In [69]:
time_thresh = 100
dist_thresh = 40
min_pts = 10

In [70]:
stops = SD.temporal_dbscan(sample_user_df, time_thresh, dist_thresh, min_pts, traj_cols=traj_cols)

In [71]:
stops.value_counts()

cluster  core
-1       -1      269
 1        1       87
 6        6       84
 3        3       57
 13       13      53
                ... 
 10      -1        2
 27      -1        1
 8       -1        1
 15      -1        1
 11      -1        1
Name: count, Length: 68, dtype: int64

In [78]:
df = filters.to_projection(df, latitude='dev_lat', longitude='dev_lon')
df['unix_timestamp'] = df['local_datetime'].astype('int64') // 10**9
sample_user_df = df[df['user_id'] == "angry_spence"]

In [79]:
stops_base = baseSD.temporal_dbscan(sample_user_df, time_thresh, dist_thresh, min_pts)

In [80]:
stops_base.value_counts()

cluster  core
-1       -1      269
 1        1       87
 6        6       84
 3        3       57
 13       13      53
                ... 
 10      -1        2
 27      -1        1
 8       -1        1
 15      -1        1
 11      -1        1
Name: count, Length: 68, dtype: int64

In [44]:
dur_min = 100
dt_max = 120
delta_roam = 10

In [45]:
SD.lachesis(sample_user_df, dur_min, dt_max, delta_roam, traj_cols=traj_cols, complete_output=False)

,start_time,duration,x,y
0,2024-01-01 18:17:00,0 days 02:09:00,-36.667413,38.320443
1,2024-01-02 09:34:00,0 days 01:43:00,-36.667709,38.320547
2,2024-01-03 14:14:00,0 days 01:54:00,-36.667298,38.320056
3,2024-01-04 21:43:00,0 days 02:50:00,-36.666779,38.320995
4,2024-01-06 23:08:00,0 days 02:10:00,-36.666982,38.321381
5,2024-01-07 15:43:00,0 days 02:00:00,-36.667278,38.321772
6,2024-01-08 03:28:00,0 days 01:44:00,-36.667393,38.321351
7,2024-01-09 00:48:00,0 days 01:59:00,-36.667506,38.321005
8,2024-01-09 04:05:00,0 days 02:54:00,-36.667502,38.320962
9,2024-01-10 03:33:00,0 days 02:05:00,-36.6674,38.320404
